# 昼夜合成反转因子：新版因子引擎示例

本 notebook 使用真实 Snapshot、SmartQuant 数据源和新版 `factor_engine` 的统一调用链计算昼夜合成反转因子：

```text
FormulaBatch → ComputeRequest → BatchFactorEngine.stream() → ResultChunk
```

计算口径：

1. 用 1 分钟收盘价计算 10:00 到日内最后一分钟的对数收益；
2. 用复权且停牌过滤后的日频开盘价、前一日收盘价计算隔夜跳空；
3. 合成日内收益 20 日均值和隔夜跳空 20 日求和；
4. 剔除过去 20 日窗口内存在不可交易记录的股票。


## 与旧 notebook 的两个口径差异

旧 notebook 创建了复权价格定义，但隔夜公式实际引用的是未复权价格；它也构造了 `tradability_mask`，但最终设置了 `output_mask=None`。本示例按照旧 notebook 的文字说明实现预期口径：显式使用 `adj_factor`，并用 `apply_mask` 应用可交易过滤。

如需复现旧代码实际数值，可以去掉 `* adj_factor`，并把最后一行改为 `factor = combined`。


## 1. 导入新版公共 API


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd


def find_project_root(start=None):
    """向上查找包含新版 src/factor_engine 包的项目根目录。"""
    start = Path.cwd() if start is None else Path(start)
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "src" / "factor_engine").exists():
            return path
    raise RuntimeError("找不到包含 src/factor_engine 的项目根目录")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from factor_engine import (
    BatchFactorEngine,
    ComputeRequest,
    DataRouter,
    DomainSpec,
    ExecutionOptions,
    FeatureStore,
    FeatureStoreDataProvider,
    FormulaBatch,
)
from factor_engine.domain import get_freq_step_values

PROJECT_ROOT


PosixPath('/data/home/dingtianxin/factor-operators-batch-engine')

## 2. 设置真实数据和计算参数

本 notebook 固定使用真实数据：日频、复权因子和可交易状态通过 SmartQuant Reader 读取，1 分钟收盘价通过 DataRouter 解析到真实分钟 Parquet。运行前需要配置数据库凭证，并确保分钟行情路径已挂载。

新版 `ExecutionOptions` 不需要手填 overlap。Compiler 会沿 Term DAG 推导 lookback，本公式当前自动得到 60 个交易日。


In [2]:
SNAPSHOT_ROOT = Path("/tmp/factor_engine_day_night_reversal2")
STORE_START = "2024-07-01"
START = "2025-01-01"
END = "2025-12-31"
INIT_SNAPSHOT = True
OVERWRITE_SNAPSHOT = False

CHUNK_SIZE = 150
OUTPUT_CSV = Path("/tmp/day_night_reversal_new_engine_full.csv")
PREVIEW_ROWS = 10
FORMULA_ID = "day_night_reversal"


## 3. 创建 DataProvider

`FeatureStore` 固定真实日期轴和股票 master axis，`DataRouter` 将逻辑数据键解析到 Store、SmartQuant 表或分钟 Parquet，`FeatureStoreDataProvider` 再把这些设施适配为新版引擎的数据端口。已有 Snapshot 默认复用；只有尚未初始化或显式设置 `OVERWRITE_SNAPSHOT=True` 时才重建。


In [3]:
store = FeatureStore(SNAPSHOT_ROOT)
snapshot_meta = store.manifest.get("snapshot") or {}
snapshot_ready = bool(snapshot_meta.get("dates_path"))

if INIT_SNAPSHOT and (OVERWRITE_SNAPSHOT or not snapshot_ready):
    store.init_snapshot(
        start=STORE_START,
        end=END,
        assets=("stk",),
        overwrite=OVERWRITE_SNAPSHOT,
    )
elif not snapshot_ready:
    raise RuntimeError("Snapshot 尚未初始化，请设置 INIT_SNAPSHOT=True")

router = DataRouter()
provider = FeatureStoreDataProvider(store, router)
snapshot_dates = store.get_dates().astype(str)

print("provider:", type(provider).__name__)
print("snapshot_root:", SNAPSHOT_ROOT)
print("snapshot date range:", snapshot_dates[0], "->", snapshot_dates[-1])
print("request date range:", START, "->", END)
print("stk asset count:", len(provider.asset_codes("stk")))


provider: FeatureStoreDataProvider
snapshot_root: /tmp/factor_engine_day_night_reversal2
snapshot date range: 20240701 -> 20251231
request date range: 2025-01-01 -> 2025-12-31
stk asset count: 5247


## 4. 检查真实数据源解析

`SourceRefExpr` 只保存稳定逻辑键；真实表、字段和 reader 参数要到每个分区的 `bind_many()` 阶段才成为 `SourceSpec`。这里提前查看 Router 解析结果只是为了人工核对字段。


In [4]:
required_keys = [
    "stk.1min.close_price",
    "stk.1d.OpenPrice",
    "stk.1d.ClosePrice",
    "stk.1d.IfSuspended",
    "stk.1d.adj_factor",
    "stk.1d.is_untradable",
]

source_df = pd.DataFrame(
    [
        {
            "key": key,
            "source": (spec := router.resolve_source(key)).source,
            "table": spec.table,
            "field": spec.field,
            "params": spec.params,
        }
        for key in required_keys
    ]
)

source_df


,key,source,table,field,params
0,stk.1min.close_price,MinuteParquet,/data/cephfs/minute/one_minute_stat/{date}.par...,close_price,"{'data_type': 'one_minute_stat', 'path_templat..."
1,stk.1d.OpenPrice,ReturnDaily,SmartQuant.ReturnDaily,OpenPrice,{}
2,stk.1d.ClosePrice,ReturnDaily,SmartQuant.ReturnDaily,ClosePrice,{}
3,stk.1d.IfSuspended,ReturnDaily,SmartQuant.ReturnDaily,IfSuspended,{}
4,stk.1d.adj_factor,AdjustFactor,JYDB.DZ_AdjustingFactor,adj_factor,{}
5,stk.1d.is_untradable,Untradable,SmartQuant.Untradable,is_untradable,{}


## 5. 把 10:00 收益表达为保持 shape 的分钟运算

当前第一版 Domain lowering 尚未把 `get_step()` 的 singleton-step 输出建模为独立 TermDomain。因此这里将 10:00 的价格沿 step 轴延迟到最后一分钟，再用 `resample(..., "1d", method="last")` 取每日最后一个比值。该写法全程保持 `T × N × 237`，最后显式降为日频。


In [5]:
step_values = get_freq_step_values("1min")
pos_1000 = int(np.where(step_values == 1000)[0][0])
shift_1000_to_close = len(step_values) - 1 - pos_1000

print("step count:", len(step_values))
print("10:00 position:", pos_1000)
print("last step:", int(step_values[-1]))
print("step-axis delay:", shift_1000_to_close)


step count: 237
10:00 position: 30
last step: 1456
step-axis delay: 206


## 6. 定义一个 FormulaBatch

新版不注册中间 FeatureDef。公共数据源放在 `inputs` 中，中间变量是单个公式程序内的顺序 binding；Compiler 会把全部表达式 lower 成一个共享 Term DAG。mask 也是显式公式语义，不放在 `ExecutionOptions` 中。


In [6]:
batch = FormulaBatch.from_text(
    common_inputs="""
        minute_close = source("stk.1min.close_price")
        open_price = source("stk.1d.OpenPrice")
        close_price = source("stk.1d.ClosePrice")
        suspended = source("stk.1d.IfSuspended")
        adj_factor = source("stk.1d.adj_factor")
        untradable = source("stk.1d.is_untradable")
    """,
    formulas={
        FORMULA_ID: f"""
            close_at_1000_shifted = delay(
                minute_close, periods={shift_1000_to_close}, axis=2
            )
            intraday_curve = ln(minute_close / close_at_1000_shifted)
            intraday_return = resample(intraday_curve, "1d", method="last")
            intraday_return_20 = ts_mean(
                intraday_return, window=20, min_periods=1
            )

            not_suspended = equal(suspended, 0)
            open_adj = apply_mask(open_price * adj_factor, not_suspended)
            close_adj = apply_mask(close_price * adj_factor, not_suspended)
            previous_close = delay(
                ts_ffill(close_adj, limit=20), periods=1, axis=0
            )
            overnight_gap = abs(ln(open_adj / previous_close))
            overnight_gap_20 = ts_sum(
                ts_ffill(overnight_gap, limit=20),
                window=20,
                min_periods=10,
            )

            untradable_count_20 = ts_sum(
                where(equal(untradable, 1), 1, 0),
                window=20,
                min_periods=1,
            )
            tradable_20 = equal(untradable_count_20, 0)
            combined = -(
                0.6 * intraday_return_20 + 0.4 * overnight_gap_20
            )
            factor = apply_mask(combined, tradable_20)
        """,
    },
)


## 7. 创建请求、结果流并 review 编译计划

`engine.stream()` 创建时完成一次编译和物理分区，但数据要到迭代结果流时才读取。我们可以先查看 LogicalPlan，再消费同一条流。


In [7]:
request = ComputeRequest(
    domain=DomainSpec(
        start=START,
        end=END,
        asset_scope={"stk": "all"},
        target_asset="stk",
        target_freq="1d",
        target_step_count=1,
    ),
    batch=batch,
)

engine = BatchFactorEngine(provider)
stream = engine.stream(request, options=ExecutionOptions(chunk_size=CHUNK_SIZE))

term_rows = []
for term_id in stream.plan.topological_order:
    term = stream.plan.terms[term_id]
    term_rows.append(
        {
            "term_id": term_id,
            "type": type(term).__name__,
            "operator": getattr(term, "operator_name", None),
            "lookback": term.lookback,
            "frequency": None if term.domain is None else term.domain.frequency,
            "steps": None if term.domain is None else term.domain.step_count,
        }
    )

plan_df = pd.DataFrame(term_rows)
print("semantic_id:", stream.plan.semantic_id)
print("job lookback:", stream.plan.job_lookback)
print("term count:", len(plan_df))
plan_df


semantic_id: 669c845ff797f8fceb1514fdc8dde570354372b918e793f97f27f12eef6dcf8a
job lookback: 60
term count: 36


,term_id,type,operator,lookback,frequency,steps
0,term_22d7d0e8fbc1cf8e,LiteralTerm,NaN,0,NaN,NaN
1,term_17df15941182ef5c,SourceTerm,NaN,0,1min,237.0
2,term_bd3c70db34f80cc5,OperatorTerm,delay,0,1min,237.0
3,term_45385c4d4c7a340d,OperatorTerm,divide,0,1min,237.0
4,term_514a520bbad576a0,OperatorTerm,ln,0,1min,237.0
5,term_45f9e32aaf8f4c0d,OperatorTerm,resample,0,1d,1.0
6,term_22656dbd7b18ad73,OperatorTerm,ts_mean,19,1d,1.0
7,term_119c4cfcf89c2947,OperatorTerm,multiply,19,1d,1.0
8,term_e8c330efdd83bd70,LiteralTerm,NaN,0,NaN,NaN
9,term_9c50ece07b77f847,SourceTerm,NaN,0,1d,1.0


## 8. 流式计算、装配完整 DataFrame 并统计覆盖率

每收到一个 `ResultChunk`，一方面计算对应日期的覆盖率，另一方面写入完整的 `date × InnerCode` 结果矩阵。最终 `factor_df` 包含请求区间内所有日期和 Snapshot 股票轴上的全部结果，包括 NaN。只有流自然结束且 `stream.succeeded=True` 时，结果才算完整成功。每个 chunk 完成后清理 Router cache，避免真实分钟数据按分区持续累积。


In [8]:
coverage_parts = []
factor_values = np.full(
    (len(stream.domain.dates), len(stream.domain.codes)),
    np.nan,
    dtype=np.float64,
)

for chunk in stream:
    values = chunk.values[:, :, 0]
    factor_values[chunk.output_slice, :] = values
    finite_count = np.isfinite(values).sum(axis=1)
    chunk_dates = stream.domain.dates[chunk.output_slice].astype(str)
    coverage_parts.append(
        pd.DataFrame(
            {
                "DataDate": chunk_dates,
                "coverage": finite_count / values.shape[1],
                "finite_count": finite_count,
            }
        )
    )
    router.clear_cache()

if not stream.succeeded:
    raise RuntimeError("因子结果流没有自然完成")

coverage_df = pd.concat(coverage_parts, ignore_index=True)
factor_df = pd.DataFrame(
    factor_values,
    index=pd.Index(stream.domain.dates.astype(str), name="DataDate"),
    columns=pd.Index(stream.domain.codes, name="InnerCode"),
)
print("load calls:", stream.stats.load_calls)
print("peak workspace values:", stream.stats.peak_workspace_values)
print("released terms:", len(stream.stats.released_terms))
print("factor_df shape:", factor_df.shape)
coverage_df.tail(PREVIEW_ROWS)


load calls: 8
peak workspace values: 7
released terms: 70
factor_df shape: (243, 5247)


,DataDate,coverage,finite_count
233,20251218,0.788641,4138
234,20251219,0.787688,4133
235,20251222,0.788451,4137
236,20251223,0.788069,4135
237,20251224,0.788260,4136
238,20251225,0.789213,4141
239,20251226,0.789785,4144
240,20251229,0.793025,4161
241,20251230,0.791309,4152
242,20251231,0.782352,4105


## 9. 输出完整结果 DataFrame，并查看最后一期截面分布

`factor_df` 是完整的宽表：index 为全部 `DataDate`，columns 为 Snapshot 的全部股票 `InnerCode`。Notebook 前端可能折叠显示，但对象本身没有只保留 head/tail。


In [9]:
display(factor_df)

target_date = str(factor_df.index[-1])
latest = factor_df.iloc[-1].rename(target_date)
latest.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])


InnerCode,3,6,14,20,23,26,28,31,34,38,...,663527,663847,664052,664089,681275,688851,701597,701706,702027,702045
DataDate,,,,,,,,,,,,,,,,,,,,,
20250102,-0.019493,-0.056843,NaN,-0.077647,-0.057010,NaN,-0.026716,-0.047201,-0.045309,-0.018973,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20250103,-0.019144,-0.057291,NaN,-0.076853,-0.054432,NaN,-0.022732,-0.052627,-0.046233,-0.018271,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20250106,-0.019178,-0.056945,NaN,-0.083234,-0.053394,NaN,-0.022323,NaN,-0.043669,-0.016449,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20250107,-0.019030,-0.057299,NaN,-0.081144,-0.055111,NaN,-0.023064,NaN,-0.042546,-0.015797,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20250108,-0.011261,-0.033425,NaN,-0.071805,-0.043462,NaN,-0.012400,NaN,-0.030409,-0.007776,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20251225,-0.007756,NaN,NaN,-0.050275,NaN,-0.020245,-0.023478,-0.042943,-0.015756,-0.020002,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20251226,-0.006414,NaN,NaN,-0.049297,NaN,-0.018230,-0.026449,-0.038781,-0.013401,-0.020565,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20251229,-0.005682,NaN,NaN,-0.049111,NaN,-0.016843,-0.025208,-0.029814,-0.013818,-0.019317,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


count    4105.000000
mean       -0.037027
std         0.022932
min        -0.191147
1%         -0.120494
5%         -0.081770
50%        -0.031176
95%        -0.012433
99%        -0.008063
max        -0.003058
Name: 20251231, dtype: float64

## 10. 生成并导出完整长表结果

将完整宽表转换为所有日期、所有股票的长表，并按日期把 Snapshot 的 `InnerCode` 映射回 `SecuCode`。`full_result_df` 保留缺失因子值和未匹配代码，便于与覆盖率结果核对；CSV 也导出这份完整长表。正式 FactorRepository 尚未设计，因此这里仍只输出 review 用文件。


In [10]:
full_result_df = pd.DataFrame(
    {
        "runner_date": np.repeat(factor_df.index.to_numpy(), factor_df.shape[1]),
        "InnerCode": np.tile(factor_df.columns.to_numpy(), factor_df.shape[0]),
        "runner_value": factor_df.to_numpy(copy=False).reshape(-1),
    }
)

code_map = store.get_code_map("stk")[["DataDate", "InnerCode", "SecuCode"]].copy()
code_map["runner_date"] = (
    code_map["DataDate"].astype(str).str.replace("-", "", regex=False).str[:8]
)
code_map = code_map.drop(columns="DataDate").drop_duplicates(
    ["runner_date", "InnerCode"], keep="last"
)
full_result_df = full_result_df.merge(
    code_map, how="left", on=["runner_date", "InnerCode"], validate="many_to_one"
)[["runner_date", "InnerCode", "SecuCode", "runner_value"]]

if OUTPUT_CSV is not None:
    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    full_result_df.to_csv(OUTPUT_CSV, index=False)
    print("saved:", OUTPUT_CSV)

print("full_result_df shape:", full_result_df.shape)
full_result_df


saved: /tmp/day_night_reversal_new_engine_full.csv
full_result_df shape: (1275021, 4)


,runner_date,InnerCode,SecuCode,runner_value
0,20250102,3,000001,-0.019493
1,20250102,6,000002,-0.056843
2,20250102,14,000004,NaN
3,20250102,20,000006,-0.077647
4,20250102,23,000007,-0.057010
...,...,...,...,...
1275016,20251231,688851,688805,NaN
1275017,20251231,701597,688807,NaN
1275018,20251231,701706,301638,NaN
1275019,20251231,702027,688802,NaN


## 调用链回顾

```text
FormulaBatch.from_text()
  → 名称绑定、helper 展开、source describe
  → Domain lowering、CSE、lookback
  → LogicalPlan
  → PhysicalPlanner 按日期分块并自动扩展 read_dates
  → DataProvider.bind_many()/load_many()
  → Runtime 拓扑执行
  → ResultStream 逐块装配完整 factor_df 并计算 coverage
  → 完整长表 full_result_df 与 CSV
```
